# aggregate current data into schema format

In [1]:
import pandas as pd
import numpy as np


### Read data

In [2]:
all_consonants = pd.read_csv('../data_processing/all_consonants_data.csv')

In [ ]:
# rename columns
all_consonants.rename(columns={'IPA key': 'IPA_key',  'Consonantal +/−': 'consonantal', 'Voice +/−': 'voice', 'Sibilant +/−': 'sibilant', 'Lateral +/−': 'lateral',
                               'Place of articulation': 'place_articulation', 'Place value': 'place_value', 'Manner of articulation': 'articulation_manner', 'Sonority value': 'consonantality_value'}, 
                     inplace=True)


### ConsonantIPA

- identify inconsistencies in original data (mislabeled IPA keys, unexpected data)

In [ ]:
# get all unique IPA/data combinations
original_IPAs = all_consonants[['IPA_key', 'IPA', 'consonantal', 'voice', 'sibilant', 'lateral', 'place_articulation', 'place_value', 'articulation_manner', 'consonantality_value']].drop_duplicates()

# remove phi keys for now
drop_keys = ['∅', None]
original_IPAs = original_IPAs[~original_IPAs['IPA_key'].isin(drop_keys)].dropna(subset = ['IPA_key'])

original_IPAs['IPA_key'] = original_IPAs['IPA_key'].astype(float)
original_IPAs.sort_values(by= ['IPA_key'], inplace=True)

original_IPAs.to_csv('original_IPAs.csv')


## compare to IPA in excel doc
    # where are there duplicates in the original full excel
    # and how many are different from IPA.csv


# duplicates in excel after finding all unique sets of data with each IPA
excel_IPAs = original_IPAs.groupby('IPA').size().reset_index(name = 'count')
excel_IPAs.sort_values('count', inplace=True)
excel_dupes = excel_IPAs[excel_IPAs['count'] > 1]

# get original IPAs where multiple versions occured
dupe_IPAs = original_IPAs[original_IPAs['IPA'].isin(excel_dupes['IPA'])].sort_values('IPA')

#dupe_IPAs.to_csv('duplicate_data.csv')
    # cleaned instances that were obviously mislabeled, subset left have very small differences, will assign data based off of IPA key and new table



- then check mapping consistency

In [ ]:
# read in full set of new IPA key data with updated column names
new_IPAs = pd.read_csv('newConsonantIPA.csv', usecols = range(11)).dropna(subset = ['key']) # remove unusued columns and rows

new_IPAs.rename(columns = {'key': 'IPA_key', 'Consonantal?': 'consonantal', 'Voiced?': 'voice', 'Sibilant?': 'sibilant', 'Lateral?': 'lateral', 'Geminate?': 'geminate',
                            'Place of\narticulation': 'place_articulation', 'Place\nvalue': 'place_value', 'Manner of\narticulation': 'articulation_manner', 'Consonantality\nvalue': 'consonantality_value'},
                inplace = True)



# get original IPA keys
og_consonant_keys = original_IPAs[['IPA', 'IPA_key']].drop_duplicates()

og_consonant_keys['IPA'].astype(str)
new_IPAs['IPA'].astype(str)

#merge old with new keys on IPA to check existence from old:new as they appear in the data
joined_keys = og_consonant_keys.merge(new_IPAs[['IPA_key', 'IPA']], how = 'right', on = 'IPA')


# check that manual mapping matches old_to_new_keys.csv
old_to_new_keys = pd.read_csv('old_to_new_keys.csv')
    # going to use old_to_new_keys as the official mapping, since not all original IPA keys show up in the data (and therefore aren't in joined_keys)

keys_test = joined_keys.merge(old_to_new_keys, how = 'right', on = 'IPA')
#keys_test.to_csv('keys_test.csv')
    # confirmed, manual mapping matches official mapping where data is present


- then add relabeled IPA keys to consonantIPA

In [12]:
consonantIPA = new_IPAs.merge(old_to_new_keys, how = 'left', right_on = 'new key', left_on = 'IPA_key')


consonantIPA = consonantIPA.iloc[:, [0, 1, 11, 2, 3, 4, 5, 6, 7, 8, 9, 10]]

consonantIPA.rename(columns = {'IPA_x': 'IPA', 'old key': 'OLD_IPA_key'}, inplace = True)


def to_binary(x):

    binary_dict = {'+': '1', '−':'0'}

    try:
        value = binary_dict[x]
    except:
        value = x

    return value
    

consonantIPA = consonantIPA.map(to_binary)


consonantIPA.to_csv('tables/consonantIPA.csv')

### ConsonantArticulation

In [14]:
consonantArticulation = pd.read_csv('exl_consonantArticulation.csv', usecols = range(3))

consonantArticulation.rename(columns = {'Consonantality values': 'articulation_manner', 'Unnamed: 1': 'consonantality_value', 'Unnamed: 2': 'description'}, inplace=True)

consonantArticulation.dropna(subset = ['articulation_manner'],inplace=True)

consonantArticulation.to_csv('tables/consonantArticulation.csv')

### VowelIPA

In [16]:
vowelIPA = pd.read_csv('exl_vowelIPA.csv')

vowelIPA.rename(columns = {'key': 'IPA_key', 'Rounded?' : 'rounded', 'Nasal?': 'nasal', 'Front?': 'front', 'Raised?': 'raised', 'Retracted?': 'retracted', 
                           'Height': 'height', 'Height\nvalue': 'height_value', 'Backness': 'backness', 'Backness\nvalue': 'backness_value'},
                           inplace=True)

vowelIPA = vowelIPA.map(to_binary).dropna(subset  = ['IPA_key'])

vowelIPA.to_csv('tables/vowelIPA.csv')


### LanguageMeta

In [ ]:
languageMeta = pd.read_csv('../languageCities.csv')

languageMeta.rename(columns = {'Language Variety': 'language_name', 'Language': 'parent_language', 'Representative City': 'rep_city'}, 
                    inplace = True)

languageMeta = languageMeta.iloc[:, [0, 5, 1, 2, 3, 4]]

languageMeta.to_csv('tables/languageMeta.csv') 